# Quest 3 → EGG → Neo4j Pipeline: Complete Agent Implementation Guide

**Purpose of this notebook**  
This notebook is a complete implementation brief for a GitHub Codespaces agent that has **no prior knowledge** of the project. It explains the target pipeline, the exact repository structure to create, the scripts to implement, the expected inputs/outputs, and the order in which to build and validate everything.

**Project context in one sentence**  
We are building a **KG-first, sensor-log-first pipeline** that turns sparse but synchronized **RGB + depth + timestamps + camera pose** observations into an **Event-Grounding Graph (EGG)** and then into **Neo4j-ready CSVs**.

**Important design decision**  
We are **not** making dense 3D reconstruction the main objective. Dense reconstruction is optional for debugging/visualization. The main goal is:

> RGB-D + pose → object states / SpatialObjects → events → EGG → Neo4j

---

## 1. EGG explained in simple language

### What the EGG paper is trying to do
EGG stands for **Event-Grounding Graph**.  
The core idea is simple:

- normal scene graphs know **what objects exist** and **where they are**
- event or caption systems know **what happened**
- EGG combines both into one graph:
  - **objects / rooms / places**
  - **events over time**
  - links showing **which objects were involved in which events**

So instead of only remembering a scene statically, EGG creates a **structured memory of object history**.

### Example
Instead of only storing:

- `Mug` is on `Table`

EGG also stores things like:

- `Event_017 = "person placed mug on table"`
- `Event_017 involves Mug`
- `Event_017 involves Table`
- `Event_017 happened between 10:04:03 and 10:04:06`
- `Mug position before event = x1,y1,z1`
- `Mug position after event = x2,y2,z2`

### Why EGG fits this project
The data we now expect is roughly:

- sparse but good RGB frames
- matching depth frames
- timestamps
- camera pose matrices

That is a very good match for an EGG-style pipeline because EGG does **not** require perfect dense 3D reconstruction as the center of the method.  
It is enough to build a graph that remembers:

- what objects were seen
- where they were
- what changed
- when it changed
- which objects participated in the change

### What we are borrowing from EGG
We are **not** copying the paper blindly.  
We are borrowing the **architecture**:

1. build object/spatial nodes from RGB-D + pose
2. build event nodes from time intervals
3. connect events to involved objects
4. allow pruning/querying of only the relevant subgraph
5. export to Neo4j for persistent storage and querying

### Important realism
The EGG paper is a **framework**, not a magical end-to-end product.  
Their implementation assumes some modules already exist, such as:
- activity moment selection
- object re-identification
- object segmentation / masks

For this project, we will build a **pragmatic baseline version first** and keep advanced components optional.

## 2. What this project will build

### Final target pipeline

```text
Quest 3 / company high-quality saved frames
+ matching depth
+ timestamps
+ intrinsics
+ camera pose matrices
    ↓
Canonical frame manifest
    ↓
Object observations / SpatialObjects
    ↓
Object linking across sparse frames
    ↓
Event windows + event summaries + object roles
    ↓
EGG graph JSON
    ↓
Pruned subgraph retrieval
    ↓
Neo4j CSV export
    ↓
Neo4j Aura import + queries
```

### What we are NOT making the main dependency
- dense reconstruction as the backbone
- real-time SLAM as the main thesis deliverable
- perfect full-video 30 fps depth pipelines
- highly complex end-to-end training at the start

### Current baseline philosophy
Build something that is:
- **scientifically defensible**
- **practical with sparse synchronized data**
- **easy to inspect and debug**
- **compatible with future improvements**

## 3. High-level implementation strategy

We will build the project in **two layers**:

### Layer A — build a robust baseline first
This layer should work even if:
- depth is only ~2 fps
- event windows are coarse
- object detection is partly manual or heuristic
- object linking is simple

This baseline should still produce:
- object nodes
- event nodes
- event-object links
- Neo4j CSVs
- usable queries

### Layer B — improve individual modules later
Possible later upgrades:
- better segmentation/detection
- stronger object re-identification
- stronger event captioning / VLM integration
- relation extraction
- task graph learning
- optional 3D visualization / denser mapping

### Rule for the agent
**Do not block the whole project because one module is not perfect.**
Always implement the simplest complete version first.

## 4. Exact assumptions to use

The agent must assume the following unless the user updates them:

### Input assumptions
For each saved high-quality frame we want, ideally:

- `frame_idx`
- `timestamp_ns`
- `rgb_path`
- `depth_path`
- `fx`, `fy`, `cx`, `cy`
- `width`, `height`
- `T_world_cam_00` ... `T_world_cam_15`
- optional `room_id`
- optional `source_stream`

### Sparse data assumption
Depth may arrive only at around **1–2 fps**, not 30 fps.

### Design consequence
Because the stream is sparse:
- do **not** build the baseline around dense TSDF reconstruction
- instead, track **object state changes between keyframes**
- store **initial and final positions for events**
- infer coarse events from object change + captions + heuristics

### Preferred intermediate representation
Use **SpatialObjects-like rows** whenever possible:

- `id`
- `x, y, z`
- `w, h, d`
- `yaw`
- label / class
- confidence
- frame index
- timestamp

## 5. Exact repository structure the agent must create

Inside the repo `XR_YOLO_Pipeline`, create this structure:

```text
XR_YOLO_Pipeline/
├── README.md
├── requirements.txt
├── .env.example
├── configs/
│   ├── pipeline.yaml
│   ├── thresholds.yaml
│   └── neo4j.yaml
├── data/
│   ├── raw/
│   │   └── session_001/
│   │       ├── rgb/
│   │       ├── depth/
│   │       ├── metadata/
│   │       └── spatialobjects/
│   ├── interim/
│   │   └── session_001/
│   └── processed/
│       └── session_001/
│           ├── manifests/
│           ├── objects/
│           ├── events/
│           ├── graphs/
│           ├── queries/
│           └── neo4j/
├── docs/
│   ├── schemas/
│   │   ├── frame_manifest.md
│   │   ├── object_observations.md
│   │   ├── events.md
│   │   └── egg_schema.md
│   └── decisions/
│       └── design_notes.md
├── notebooks/
│   └── Quest3_EGG_Pipeline_Agent_Implementation_Guide.ipynb
├── scripts/
│   ├── 00_bootstrap_repo.py
│   ├── 01_build_frame_manifest.py
│   ├── 02_validate_manifest.py
│   ├── 03_visualize_rgb_depth_pose.py
│   ├── 04_ingest_spatialobjects.py
│   ├── 05_build_object_observations.py
│   ├── 06_link_object_tracks.py
│   ├── 07_build_event_windows.py
│   ├── 08_generate_event_summaries.py
│   ├── 09_build_egg_graph.py
│   ├── 10_prune_egg_graph.py
│   ├── 11_export_neo4j_csv.py
│   ├── 12_demo_queries.py
│   └── 13_visualize_3d_debug.py
├── src/
│   ├── __init__.py
│   ├── config.py
│   ├── io_utils.py
│   ├── geometry.py
│   ├── pose_utils.py
│   ├── depth_utils.py
│   ├── objects.py
│   ├── tracking.py
│   ├── events.py
│   ├── egg.py
│   ├── pruning.py
│   ├── neo4j_export.py
│   └── viz.py
└── tests/
    ├── test_manifest.py
    ├── test_geometry.py
    ├── test_events.py
    └── test_egg.py
```

### Important note
The agent must build **the scripts and the reusable `src/` modules together**.
Do not place all logic in notebooks.

## 6. Environment setup the agent should implement

Use Python first. Do not start with a heavy C++ stack.

### Recommended baseline dependencies
Put this in `requirements.txt`:

```text
pandas
numpy
pydantic
pyyaml
networkx
open3d
opencv-python
pillow
matplotlib
rich
typer
python-dotenv
neo4j
scipy
```

### Codespaces setup commands
Use these commands in Codespaces:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -U pip
pip install -r requirements.txt
```

### Why this stack
- `pandas` / `numpy`: table and numeric work
- `pydantic`: strong schema validation
- `networkx`: easy graph construction before Neo4j export
- `open3d`: visualization, point cloud lifting, debug only
- `opencv-python` / `pillow`: image IO
- `neo4j`: optional direct database checks
- `typer`: clean CLI scripts
- `rich`: clean logs

### Rule
The first working version should not require cloud APIs.
If event captioning is unavailable, use:
- rule-based summaries first
- placeholders second
- VLM integration later

## 7. Data contract the agent must enforce

Before anything else, the agent must create and validate a **canonical frame manifest**.

### Canonical frame manifest CSV
Output file:

```text
data/processed/session_001/manifests/frame_manifest.csv
```

### Required columns
Use these exact columns:

| column | meaning |
|---|---|
| frame_idx | integer frame index |
| timestamp_ns | timestamp in nanoseconds |
| rgb_path | relative path to RGB image |
| depth_path | relative path to depth image or encoded depth source |
| depth_encoding | e.g. png16, npy, alpha_channel, unknown |
| depth_scale | scaling factor from stored value to meters |
| fx | focal length x |
| fy | focal length y |
| cx | principal point x |
| cy | principal point y |
| width | image width |
| height | image height |
| T_world_cam_00 ... T_world_cam_15 | flattened 4x4 pose matrix row-major |
| room_id | optional room / workstation id |
| source_stream | e.g. sampled_hq or webrtc |
| notes | optional note / warning |

### Very important questions to resolve with the supervisor/company pipeline
If not already known, the agent must leave placeholders and document these questions:

1. Is depth stored as a separate image or in the alpha channel?
2. If it is in alpha, what is the exact encoding and scale?
3. Is the depth frame aligned with the RGB frame?
4. Are the camera pose matrices timestamped for the same saved frames?
5. What coordinate frame convention is used?
6. Is the saved frame stream the same stream that carries depth?
7. Are intrinsics fixed or frame-dependent?

### Validation rule
Do not proceed to downstream modules until:
- `frame_manifest.csv` loads successfully
- every row has a readable RGB path
- every row has either readable depth or explicitly marked missing depth
- every row has a valid 4x4 pose matrix

## 8. The exact staged build order

The agent must implement the project in this order.

### Stage 0 — bootstrap
Create the repo structure, configs, schemas, and CLI skeletons.

### Stage 1 — frame manifest and data QA
Goal: standardize all incoming data into one manifest.

Outputs:
- `frame_manifest.csv`
- validation report
- sample visualizations

### Stage 2 — object observations
Goal: create per-frame object observations from either:
- existing `spatialobjects.csv` / `spatialobjects_clean.csv`, or
- masks / detections / manual annotations

Outputs:
- `object_observations.csv`

### Stage 3 — object linking
Goal: connect same-object observations across sparse frames.

Outputs:
- `object_tracks.csv`

### Stage 4 — event windows
Goal: identify coarse event intervals from frame-to-frame changes.

Outputs:
- `event_windows.csv`

### Stage 5 — event summaries and roles
Goal: attach a human-readable summary and object roles to each event.

Outputs:
- `events.csv`
- `event_object_roles.csv`

### Stage 6 — build EGG
Goal: convert objects, rooms, events, and links into one graph JSON.

Outputs:
- `egg_graph.json`

### Stage 7 — prune and query
Goal: retrieve a task-specific subgraph.

Outputs:
- `pruned_subgraph.json`
- query answers

### Stage 8 — export Neo4j CSV
Goal: create import-ready CSV files.

Outputs:
- `nodes_objects.csv`
- `nodes_events.csv`
- `nodes_rooms.csv`
- `edges_event_object.csv`
- `edges_room_object.csv`
- `edges_before.csv`
```

## 9. Stage 0 in detail — bootstrap the project

### What the agent must implement

#### `scripts/00_bootstrap_repo.py`
Responsibilities:
- create missing folders
- create default configs if absent
- create empty schema docs if absent
- create a sample `.env.example`
- print next steps

#### `src/config.py`
Responsibilities:
- load YAML config files
- expose paths and thresholds

#### `configs/pipeline.yaml`
Must define:
- session id
- raw data root
- processed data root
- whether depth is separate or alpha-encoded
- whether to start from `spatialobjects.csv`
- whether event summaries are rule-based or VLM-based

#### `configs/thresholds.yaml`
Must define initial thresholds such as:
- max spatial jump for track linking
- max time gap for track linking
- min movement distance for `MOVE`
- min relation duration
- event merge gap
- confidence thresholds

### Expected command
```bash
python scripts/00_bootstrap_repo.py
```

### Expected result
The repo becomes ready for implementation even before sample data arrives.

## 10. Stage 1 in detail — build and validate the canonical frame manifest

### Script 1: `scripts/01_build_frame_manifest.py`
Purpose:
- scan the raw session folder
- match RGB, depth, metadata, and poses
- produce `frame_manifest.csv`

### Script 2: `scripts/02_validate_manifest.py`
Purpose:
- verify file existence
- verify pose matrix shape
- verify timestamps are monotonic
- verify image sizes
- verify depth decoding can succeed on at least a sample

### Script 3: `scripts/03_visualize_rgb_depth_pose.py`
Purpose:
- draw sample RGB frames
- draw decoded depth frames
- print pose translation values
- optionally save a quick point cloud for a few frames

### Exact outputs
```text
data/processed/session_001/manifests/frame_manifest.csv
data/processed/session_001/manifests/manifest_validation.json
data/processed/session_001/manifests/sample_visualizations/
```

### Very important instruction
This stage is where the agent must solve the practical issue:
- separate depth file
- or alpha-channel depth
- or no usable depth yet

If alpha-channel depth is used, the agent must create a reusable decoder in:
- `src/depth_utils.py`

### Minimal success condition
You can open at least 3 sample frames and confirm:
- RGB looks correct
- depth decodes without nonsense values
- pose matrices vary in a plausible way

## 11. Stage 2 in detail — build object observations

This stage should support **two entry paths**.

### Path A — start from existing SpatialObjects CSV
If you already have:
- `spatialobjects.csv`
- `spatialobjects_clean.csv`

then the agent should use that first because it is the fastest path to a working graph.

#### Script: `scripts/04_ingest_spatialobjects.py`
Input:
```text
data/raw/session_001/spatialobjects/spatialobjects.csv
data/raw/session_001/spatialobjects/spatialobjects_clean.csv
```

Output:
```text
data/processed/session_001/objects/object_observations.csv
```

### Path B — derive object observations from RGB-D
If no SpatialObjects CSV exists, use:
- masks
- boxes
- segmented point clouds
- lifted centroids / extents

#### Script: `scripts/05_build_object_observations.py`
This script must:
1. load one frame at a time
2. decode RGB + depth + intrinsics + pose
3. use either provided detections/masks or a placeholder adapter
4. compute object-level 3D estimates
5. write one row per object observation

### Required `object_observations.csv` columns
Use these exact columns:

| column | meaning |
|---|---|
| observation_id | unique observation id |
| frame_idx | frame index |
| timestamp_ns | timestamp |
| raw_object_id | raw source id if available |
| track_hint | optional prior track id |
| semantic_class | class name |
| label | display label |
| confidence | numeric confidence |
| x | object center x in world coordinates |
| y | object center y in world coordinates |
| z | object center z in world coordinates |
| w | object width |
| h | object height |
| d | object depth |
| yaw | object yaw |
| room_id | optional room/workstation |
| caption | optional visual description |
| source | spatialobjects_csv / detection / manual |
| mask_path | optional path |
| notes | optional warning |

### Rule
If a dimension cannot be estimated yet, leave it null and document it.  
Do not block the rest of the pipeline.

## 12. Stage 3 in detail — link object observations into object tracks

Sparse streams are difficult, so keep the first tracker simple.

### Script: `scripts/06_link_object_tracks.py`

### Goal
Convert per-frame observations into persistent object identities across time.

### Baseline linking logic
The first version should link observations using:
1. semantic class agreement
2. spatial proximity in world coordinates
3. size similarity
4. time gap threshold
5. optional caption similarity

### Output
```text
data/processed/session_001/objects/object_tracks.csv
```

### Required columns
| column | meaning |
|---|---|
| track_id | stable object identity |
| observation_id | linked observation |
| frame_idx | frame index |
| timestamp_ns | timestamp |
| semantic_class | class |
| x,y,z,w,h,d,yaw | spatial state |
| is_first_in_track | bool |
| is_last_in_track | bool |
| linkage_score | numeric score |

### Important caution
At ~2 fps, fine hand motion may be missed.  
So this tracker should aim for:
- stable coarse identities
- not perfect frame-by-frame motion

### Debug outputs
The script must also write:
```text
data/processed/session_001/objects/track_summary.csv
data/processed/session_001/objects/track_debug.json
```

### Minimum success condition
For a small sample session, a human can inspect:
- how many tracks exist
- whether obvious objects keep the same id over time

## 13. Stage 4 in detail — build event windows

This is where we shift from **objects** to **what happened**.

### Script: `scripts/07_build_event_windows.py`

### Goal
Group object changes over time into coarse event intervals.

### Baseline event triggers
The first version should support rule-based triggers from sparse frames:

- `MOVE`: object position changed above threshold
- `APPEAR`: object newly visible
- `DISAPPEAR`: object no longer visible
- `CO_LOCATE`: two objects become near or overlapping
- `SEPARATE`: two objects stop being near
- `STATE_CHANGE_CANDIDATE`: object role or relation changed
- `ASSEMBLY_CHANGE_CANDIDATE`: known relation/constraint changed

### Important recommendation
At 2 fps, do **not** try to detect very fine-grained hand actions first.
Instead start with **object-centered change events**.

### Output
```text
data/processed/session_001/events/event_windows.csv
```

### Required columns
| column | meaning |
|---|---|
| event_id | unique event id |
| event_type | coarse event label |
| start_frame_idx | first frame |
| end_frame_idx | last frame |
| start_ts_ns | start time |
| end_ts_ns | end time |
| primary_track_ids | JSON array of involved track ids |
| room_id | room/workstation |
| trigger_reason | why the event was created |
| confidence | rule confidence |

### Recommended design
Store events first as **coarse windows**.  
Human-readable summaries come in the next stage.

## 14. Stage 5 in detail — event summaries and object roles

### Script: `scripts/08_generate_event_summaries.py`

### Goal
Turn a coarse event window into:
- a summary sentence
- a per-object role description

### First version strategy
Use deterministic templates first.

#### Example
If:
- object A moved
- object A ended near object B

then summary could be:
- `"Object A was moved near Object B."`

roles could be:
- `A`: `"moving object"`
- `B`: `"target/related object"`

### Optional later upgrade
Replace or enrich the template system with:
- Moondream outputs
- another VLM
- manually supplied captions

### Outputs
```text
data/processed/session_001/events/events.csv
data/processed/session_001/events/event_object_roles.csv
```

### Required `events.csv` columns
| column | meaning |
|---|---|
| event_id | unique event id |
| event_type | coarse type |
| summary | one-sentence event summary |
| start_ts_ns | start |
| end_ts_ns | end |
| room_id | room/workstation |
| event_pos_x | event anchor x |
| event_pos_y | event anchor y |
| event_pos_z | event anchor z |
| source | rule / vlm / manual |
| confidence | confidence |

### Required `event_object_roles.csv` columns
| column | meaning |
|---|---|
| event_id | event id |
| track_id | object track id |
| role | object role in event |
| role_description | natural-language explanation |

### Important note
For EGG-style grounding, a simple summary + object roles is already enough for the first graph.

## 15. Stage 6 in detail — build the EGG graph

### Script: `scripts/09_build_egg_graph.py`

### Goal
Create one unified graph from:
- rooms / workstations
- object tracks
- event nodes
- room-object edges
- event-object edges
- temporal ordering edges

### Output
```text
data/processed/session_001/graphs/egg_graph.json
```

### JSON structure to use

```json
{
  "graph_metadata": {
    "session_id": "session_001",
    "created_at": "ISO_TIMESTAMP",
    "source": "quest/company_sparse_rgbd_pose",
    "notes": []
  },
  "rooms": [
    {
      "room_id": "workstation_A",
      "name": "workstation_A",
      "position": {"x": 0.0, "y": 0.0, "z": 0.0}
    }
  ],
  "objects": [
    {
      "track_id": "obj_0001",
      "semantic_class": "container",
      "label": "blue container",
      "caption": "blue plastic container",
      "time_invariant": {},
      "time_variant_history": [
        {
          "timestamp_ns": 0,
          "frame_idx": 0,
          "x": 0.0, "y": 0.0, "z": 0.0,
          "w": 0.1, "h": 0.1, "d": 0.1,
          "yaw": 0.0
        }
      ]
    }
  ],
  "events": [
    {
      "event_id": "evt_0001",
      "event_type": "MOVE",
      "summary": "Container moved near lid.",
      "start_ts_ns": 0,
      "end_ts_ns": 1000000000,
      "position": {"x": 0.0, "y": 0.0, "z": 0.0}
    }
  ],
  "event_edges": [
    {
      "event_id": "evt_0001",
      "track_id": "obj_0001",
      "role": "moving_object",
      "role_description": "the object that moved"
    }
  ],
  "room_edges": [
    {
      "room_id": "workstation_A",
      "track_id": "obj_0001"
    }
  ],
  "temporal_edges": [
    {
      "src_event_id": "evt_0001",
      "dst_event_id": "evt_0002",
      "relation": "BEFORE"
    }
  ]
}
```

### Important instruction
This graph must be serializable, inspectable, and easy to prune.
Do not make it overly clever.

## 16. Stage 7 in detail — EGG pruning and query answering

### Script: `scripts/10_prune_egg_graph.py`

### Goal
Given a query, retrieve only the relevant subgraph.

### Use the EGG idea in a pragmatic way
Support these filters:
- time interval
- room/workstation
- object names / classes
- event types
- history expansion for selected objects

### Input examples
```text
Where was the blue container last seen?
What object was moved near the lid?
Which events happened on workstation_A between 10:00 and 10:15?
```

### Output files
```text
data/processed/session_001/queries/pruned_subgraph.json
data/processed/session_001/queries/query_answer.json
```

### First implementation rule
Use deterministic filtering first.
Do not require an LLM in the first working version.

### Required helper modules
- `src/pruning.py`
- `src/egg.py`

### Minimum success condition
The user can ask a small set of structured questions and get correct answers from the graph.

## 17. Stage 8 in detail — export to Neo4j CSV

### Script: `scripts/11_export_neo4j_csv.py`

### Goal
Convert the EGG into CSV files that are easy to import into Neo4j Aura.

### Output folder
```text
data/processed/session_001/neo4j/
```

### Required CSV files

#### Node CSVs
- `nodes_rooms.csv`
- `nodes_objects.csv`
- `nodes_events.csv`

#### Edge CSVs
- `edges_room_object.csv`
- `edges_event_object.csv`
- `edges_before.csv`

### Exact recommended columns

#### `nodes_rooms.csv`
| room_id:ID(Room) | name | x:float | y:float | z:float | :LABEL |
|---|---|---:|---:|---:|---|
| workstation_A | workstation_A | 0 | 0 | 0 | Room |

#### `nodes_objects.csv`
| track_id:ID(Object) | semantic_class | label | caption | :LABEL |
|---|---|---|---|---|
| obj_0001 | container | blue container | blue plastic container | Object |

#### `nodes_events.csv`
| event_id:ID(Event) | event_type | summary | start_ts_ns:long | end_ts_ns:long | pos_x:float | pos_y:float | pos_z:float | :LABEL |
|---|---|---|---:|---:|---:|---:|---:|---|
| evt_0001 | MOVE | Container moved near lid. | 0 | 1000000000 | 0.1 | 0.2 | 0.3 | Event |

#### `edges_room_object.csv`
| :START_ID(Room) | :END_ID(Object) | :TYPE |
|---|---|---|
| workstation_A | obj_0001 | CONTAINS |

#### `edges_event_object.csv`
| :START_ID(Event) | :END_ID(Object) | role | role_description | :TYPE |
|---|---|---|---|---|
| evt_0001 | obj_0001 | moving_object | the object that moved | INVOLVES |

#### `edges_before.csv`
| :START_ID(Event) | :END_ID(Event) | :TYPE |
|---|---|---|
| evt_0001 | evt_0002 | BEFORE |

### Rule
Use stable ids and consistent labels.  
Do not use random ids that change between runs.

## 18. Neo4j Aura import guidance

Create a folder:
```text
neo4j/
```

Inside it create:
```text
neo4j/import_egg.cypher
```

### Put these Cypher commands there

```cypher
CREATE CONSTRAINT room_id IF NOT EXISTS
FOR (r:Room) REQUIRE r.room_id IS UNIQUE;

CREATE CONSTRAINT object_id IF NOT EXISTS
FOR (o:Object) REQUIRE o.track_id IS UNIQUE;

CREATE CONSTRAINT event_id IF NOT EXISTS
FOR (e:Event) REQUIRE e.event_id IS UNIQUE;
```

Then create import queries like:

```cypher
LOAD CSV WITH HEADERS FROM $rooms_url AS row
MERGE (r:Room {room_id: row.`room_id:ID(Room)`})
SET r.name = row.name,
    r.x = toFloat(row.`x:float`),
    r.y = toFloat(row.`y:float`),
    r.z = toFloat(row.`z:float`);

LOAD CSV WITH HEADERS FROM $objects_url AS row
MERGE (o:Object {track_id: row.`track_id:ID(Object)`})
SET o.semantic_class = row.semantic_class,
    o.label = row.label,
    o.caption = row.caption;

LOAD CSV WITH HEADERS FROM $events_url AS row
MERGE (e:Event {event_id: row.`event_id:ID(Event)`})
SET e.event_type = row.event_type,
    e.summary = row.summary,
    e.start_ts_ns = toInteger(row.`start_ts_ns:long`),
    e.end_ts_ns = toInteger(row.`end_ts_ns:long`),
    e.pos_x = toFloat(row.`pos_x:float`),
    e.pos_y = toFloat(row.`pos_y:float`),
    e.pos_z = toFloat(row.`pos_z:float`);

LOAD CSV WITH HEADERS FROM $room_object_url AS row
MATCH (r:Room {room_id: row.`:START_ID(Room)`})
MATCH (o:Object {track_id: row.`:END_ID(Object)`})
MERGE (r)-[:CONTAINS]->(o);

LOAD CSV WITH HEADERS FROM $event_object_url AS row
MATCH (e:Event {event_id: row.`:START_ID(Event)`})
MATCH (o:Object {track_id: row.`:END_ID(Object)`})
MERGE (e)-[rel:INVOLVES]->(o)
SET rel.role = row.role,
    rel.role_description = row.role_description;

LOAD CSV WITH HEADERS FROM $before_url AS row
MATCH (e1:Event {event_id: row.`:START_ID(Event)`})
MATCH (e2:Event {event_id: row.`:END_ID(Event)`})
MERGE (e1)-[:BEFORE]->(e2);
```

### Important note
If you import into Aura from the web, the CSVs usually need to be hosted somewhere reachable by URL or imported through the Aura tooling. Keep the exported CSVs clean and stable.

## 19. 3D visualization guidance

Even though dense reconstruction is not the main goal, the agent must still create basic 3D debug tools.

### Script: `scripts/13_visualize_3d_debug.py`

### Responsibilities
- load a few frames
- decode RGB + depth
- create sparse point clouds with Open3D
- overlay object boxes / centroids if available
- save screenshots or visual snapshots

### Why this matters
This is how we check:
- whether depth decoding is correct
- whether pose matrices make sense
- whether object centers/boxes are plausible
- whether the pipeline is grounding objects to the right place

### Outputs
```text
data/processed/session_001/graphs/debug_pointclouds/
data/processed/session_001/graphs/debug_boxes/
```

### Rule
Use 3D visualization for **verification**, not as the main representation.

## 20. Suggested baseline logic for object relations and assembly-state reasoning

This part is especially useful if the project wants an "assembly graph" inside the KG.

### Start with simple geometric relations
For each event window or frame pair, compute:

- `NEAR`
- `FAR`
- `LEFT_OF`
- `RIGHT_OF`
- `ABOVE`
- `BELOW`
- `OVERLAPS`
- `CONTACT_CANDIDATE`

### Then derive assembly-oriented state changes
Examples:

- if two parts become `NEAR` and stay stable → `ALIGNED_CANDIDATE`
- if relative pose stabilizes near a known relation → `ATTACHED_CANDIDATE`
- if object A repeatedly appears near object B during tool-use windows → `USES_TOOL_WITH`
- if object enters same region and remains there → `PLACED_AT`

### Very important recommendation
Do not start with very ambitious labels like:
- `tightening_screw`
- `fastening_with_torque`
unless the data truly supports it.

With sparse 2 fps data, the baseline should emphasize:
- object state changes
- relation changes
- coarse procedure steps

### Good first event vocabulary
Use this initial event set:
- `APPEAR`
- `DISAPPEAR`
- `MOVE`
- `PLACE`
- `ALIGN_CANDIDATE`
- `ATTACH_CANDIDATE`
- `SEPARATE`
- `INSPECT`
- `USE_TOOL_CANDIDATE`

## 21. Minimum viable deliverable and advanced roadmap

### Minimum viable deliverable (must exist)
By the end of the first working pass, the repo must be able to do this end-to-end:

1. load sparse RGB-D + pose data
2. build `frame_manifest.csv`
3. ingest or derive object observations
4. link them into object tracks
5. build event windows
6. generate simple event summaries and roles
7. construct `egg_graph.json`
8. export Neo4j CSVs
9. answer at least 3 demo queries

### Advanced improvements (later)
Only after the baseline works:
- better segmentation models
- better re-identification
- VLM event summaries
- stronger spatial reasoning
- task graph learning
- richer assembly-state logic
- optional denser 3D mapping

### Thesis-safe interpretation
The baseline is already scientifically meaningful because it demonstrates:
- spatial grounding
- event grounding
- structured graph construction
- graph querying
- Neo4j export

## 22. Acceptance checklist for the agent

The agent must not claim success until all boxes below are truly satisfied.

### Data layer
- [ ] `frame_manifest.csv` exists
- [ ] RGB loads correctly
- [ ] depth decodes correctly on sample frames
- [ ] pose matrices are valid 4x4 matrices
- [ ] manifest validation report exists

### Object layer
- [ ] `object_observations.csv` exists
- [ ] object positions are plausible
- [ ] at least a few stable tracks exist in `object_tracks.csv`

### Event layer
- [ ] `event_windows.csv` exists
- [ ] `events.csv` exists
- [ ] `event_object_roles.csv` exists
- [ ] summaries are readable and consistent

### Graph layer
- [ ] `egg_graph.json` exists
- [ ] the graph loads and serializes without error
- [ ] `pruned_subgraph.json` can be created from a query

### Neo4j layer
- [ ] CSV export folder exists
- [ ] all required node and edge CSVs exist
- [ ] import Cypher file exists

### Debug layer
- [ ] sample 3D visualizations exist
- [ ] at least 3 screenshots or outputs confirm the geometry is plausible

## 23. Exact implementation order the agent should follow in Codespaces

Run these steps in order:

```bash
python scripts/00_bootstrap_repo.py
python scripts/01_build_frame_manifest.py --session session_001
python scripts/02_validate_manifest.py --session session_001
python scripts/03_visualize_rgb_depth_pose.py --session session_001
python scripts/04_ingest_spatialobjects.py --session session_001
python scripts/05_build_object_observations.py --session session_001
python scripts/06_link_object_tracks.py --session session_001
python scripts/07_build_event_windows.py --session session_001
python scripts/08_generate_event_summaries.py --session session_001
python scripts/09_build_egg_graph.py --session session_001
python scripts/10_prune_egg_graph.py --session session_001 --query "Where was the blue container last seen?"
python scripts/11_export_neo4j_csv.py --session session_001
python scripts/13_visualize_3d_debug.py --session session_001
```

### Note
If `spatialobjects.csv` already exists and is good, `scripts/05_build_object_observations.py` can act mainly as a normalizer rather than a detector.

## 24. Final recommendation for this project

If the incoming data really is:
- sparse but high-quality keyframes
- matching depth
- timestamps
- pose matrices

then this EGG-style pipeline is a very strong fit.

### Why
Because it lets you focus on:
- grounded object memory
- event history
- assembly/procedure reasoning
- knowledge graph export

instead of getting blocked by dense 3D reconstruction.

### The key mindset
Think of this project as:

> **structured procedural memory from grounded sensor observations**

not as:

> **rebuild the whole scene perfectly in 3D**

That mindset is what makes the pipeline realistic and thesis-friendly.